In [1]:
import os
import sys
sys.path.append("/home/zhangsd/repos/CF-BGAP")
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
from GCMC.gcmc_process import create_tasks, run_and_check_pre_tasks
from GCMC.utils import run_simulation

from GCMC.utils import process_isotherm_results, process_heat_results
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn import metrics

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


## GCMC run

In [2]:
user_defined_params = {
        "NumberOfAdsorptionCycles": 5000,      # Number of adsorption MC cycles
        "NumberOfAdsorptionInitCycles": 5000,  # Number of adsorption MC cycles for initialization
        "NumberOfAdsorptionEqCycles": 0,  # Number of adsorption MC cycles for equilibration
        "NumberOfHeliumVFCycles": 5000,  # Number of MC cycles for helium void fraction calculation
        "NumberOfQstInitCycles": 5000,  # Number of Qst MC cycles for initialization
        "NumberOfQstCycles": 5000,  # Number of Qst MC cycles for calculation
        "PrintEvery": 100,            # Print results every N cycles    
        "RestartFile": "no",          # Restart from previous simulation
        "RemoveAtomNumberCodeFromLabel": "yes",  # Remove atom number code from label in input cif file

        "Forcefield": "UFF",          # Forcefield to use        
        "UseChargesFromCIFFile": "yes",  # Use charges from CIF file
        "ChargeMethod": "Ewald",      # Charge method to use
        "CutOff": 12.8,               # Cut-off distance for Ewald summation
        "EwaldPrecision": 1e-6,       # Precision for Ewald summation
        "TimeStep": 0.0005,           # Time step for MC simulation
        "CutOffRule": "shifted",    # Cut-off rule for non-bonded interactions: "truncated", "shifted"
        "TailCorrection": "yes",       # Use tail correction for non-bonded interactions

        "ExternalTemperature": 298.0,  # External temperature
        "ExternalPressure":  [round(0.1*100000, 7) , round(100000, 7)], # External pressure
        "PressureMode": "parallel",   # Pressure mode: "parallel", "sequential"

        "AllComponents": ["CO2", "N2"],  # Components to simulate
        "MoleculeDefinition": "TraPPE",  # Molecule definition
        "TranslationProbability": 0.5,  # Probability of translation
        "ReinsertionProbability": 0.5,  # Probability of reinsertion
        "RotationProbability": 0.5,  # Probability of rotation    
        "SwapProbability": 1.0,  # Probability of swap
        "CreateNumberOfMolecules": 0,  # Number of molecules to create

        "MolFractions": [[0.15, 0.85], ],  # Mole fractions of components
        "IdentityChangesList": [0, 1],  # List of identity changes
        "NumberOfIdentityChanges": 2,  # Number of identity changes

        "AdsorptionComponetTypes": ["mixture"],  # Adsorption component type: "pure", "mixture"
        "QstMethods": [],  # Qst methods to use: "widom", "ideal", "fluctuate"
        "ChargeEqMethod": "Qeq",  # Charge equilibration method to use: None, "EQeq", "Qeq"
        "CalHeliumVF": True,  # Calculate helium void fraction

        "TaskPrefix": "stableMOF",  # Prefix for workdir name
        "TaskExecutor": "bash", # Task executor: "sbatch", "sbatch_array", "bash", or, "nohup"
        "RandomSeed": 42,  # Random seed for reproducibility
    }
user_defined_params["ExternalPressure"]

[10000.0, 100000]

In [3]:
def gcmc_pre_task_submit(workdir, n_cpus=180, nodelist=None, job_name="gcmc"):
    exec_file = Path("/home/zhangsd/repos/CF-BGAP/GCMC/gcmc_pre_task_run.py").absolute()
    pre_task_slurm_template = """#!/bin/bash
    #SBATCH --job-name={job_name}
    #SBATCH --output=%x_%A.out
    #SBATCH --error=%x_%A.err
    #SBATCH --partition=C9654 
    #SBATCH --nodelist={nodelist}
    #SBATCH --ntasks=1
    #SBATCH --ntasks-per-node=1
    #SBATCH --cpus-per-task={num_cpus}
    #SBATCH --nodes=1
    export PATH=/opt/share/miniconda3/envs/lammps/bin/:$PATH
    export LD_LIBRARY_PATH=/opt/share/miniconda3/envs/lammps/lib/:$LD_LIBRARY_PATH

    python -u {exec_file} --workdir {workdir} --n_cpus {num_cpus}"""

    pre_task_slurm_template = "\n".join([line.strip() for line in pre_task_slurm_template.split("\n")])

    workdir = Path(workdir).absolute()
    if nodelist is None:
        pre_task_slurm_template = pre_task_slurm_template.replace("#SBATCH --nodelist={nodelist}\n", "")
        with open(workdir/"slurm_pre_task.sh", "w") as f:
            f.write(pre_task_slurm_template.format(
                job_name=job_name,
                num_cpus=n_cpus,
                exec_file=exec_file,
                workdir=workdir,
                ))
    else:
        with open(workdir/"slurm_pre_task.sh", "w") as f:
            f.write(pre_task_slurm_template.format(
                job_name=job_name,
                num_cpus=n_cpus,
                nodelist=nodelist,
                exec_file=exec_file,
                workdir=workdir,
                ))
    process = run_simulation(workdir.absolute(), executor="sbatch", script_name="slurm_pre_task.sh")
    # ## get the output of the job
    while True:
        output = process.stdout.readline()
        err = process.stderr.readline()
        if err:
            print(err.decode().strip())
            break
        if output == b'' and process.poll() is not None:
            break
        if output:
            print(output.decode().strip())

def gcmc_task_submit(workdir, min_press=0, max_press=10000000, n_cpus=180, nodelist=None, job_name="gcmc"):
    exec_file = Path("/home/zhangsd/repos/CF-BGAP/GCMC/gcmc_task_run.py").absolute()
    task_slurm_template = """#!/bin/bash
    #SBATCH --job-name={job_name}
    #SBATCH --output=%x_%A.out
    #SBATCH --error=%x_%A.err
    #SBATCH --partition=C9654 
    #SBATCH --nodelist={nodelist}
    #SBATCH --ntasks=1
    #SBATCH --ntasks-per-node=1
    #SBATCH --cpus-per-task={num_cpus}
    #SBATCH --nodes=1
    export PATH=/opt/share/miniconda3/envs/lammps/bin/:$PATH
    export LD_LIBRARY_PATH=/opt/share/miniconda3/envs/lammps/lib/:$LD_LIBRARY_PATH

    python -u {exec_file} --workdir {workdir} --n_cpus {num_cpus} --max_press {max_press} --min_press {min_press}"""

    script_name = "slurm_task_{}_{}.sh".format(min_press, max_press)
    task_slurm_template = "\n".join([line.strip() for line in task_slurm_template.split("\n")])
    workdir = Path(workdir).absolute()
    if nodelist is None:
        task_slurm_template = task_slurm_template.replace("#SBATCH --nodelist={nodelist}\n", "")
        with open(workdir/script_name, "w") as f:
            f.write(task_slurm_template.format(
                exec_file=exec_file,
                job_name=job_name,
                num_cpus=n_cpus,
                max_press=max_press,
                min_press=min_press,
                workdir=workdir,
            ))
    else:
        with open(workdir/script_name, "w") as f:
                f.write(task_slurm_template.format(
                    exec_file=exec_file,
                    job_name=job_name,
                    nodelist=nodelist,
                    num_cpus=n_cpus,
                    max_press=max_press,
                    min_press=min_press,
                    workdir=workdir,
                ))
            
    process = run_simulation(workdir.absolute(), executor="sbatch", script_name=script_name)
    # ## get the output of the job
    while True:
        output = process.stdout.readline()
        err = process.stderr.readline()
        if err:
            print(err.decode().strip())
            break
        if output == b'' and process.poll() is not None:
            break
        if output:
            print(output.decode().strip())

In [7]:
source_cif_dir = Path("/home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606")
gcmc_data_dir = Path(f"/home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc{user_defined_params['ExternalTemperature']}K").absolute()
gcmc_cif_dir = source_cif_dir
# if gcmc_cif_dir.exists():
#     shutil.rmtree(gcmc_cif_dir)
gcmc_cif_dir.mkdir(exist_ok=True, parents=True)
mof_list = [f.stem for f in source_cif_dir.glob("*.cif")]
print(gcmc_data_dir)

gcmc_workdir = gcmc_data_dir/"mc_data"
print("workdir: {}".format(gcmc_workdir))

/home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc298.0K
workdir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc298.0K/mc_data


In [13]:
384*3/46

25.043478260869566

In [15]:
gcmc_workdir = gcmc_data_dir/"mc_data"
print("workdir: {}".format(gcmc_workdir))

user_defined_params["TaskPrefix"] = "coremof"

## create tasks
batch_size = 100
batch_num = np.ceil(len(mof_list) / batch_size).astype(int)-1
print("batch_num: {}".format(batch_num))
## create tasks
for i in range(batch_num):
    if i == batch_num-1:
        sub_samples = mof_list[i * batch_size:]
    else:
        sub_samples = mof_list[i * batch_size: (i+1)*batch_size]
    sub_gcmc_workdir = gcmc_workdir/f"batch{i}"
    task_dirs, task_tuples = create_tasks(sub_gcmc_workdir, 
                                        sub_samples, 
                                        gcmc_cif_dir, 
                                        user_defined_params, 
                                        overwrite=True,
                                        verbose=1
                                        )

workdir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc323.0K/mc_data
batch_num: 46
sub dir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc323.0K/mc_data/batch0/CUTFON_charged
sub dir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc323.0K/mc_data/batch0/WUQRAC_clean_h
sub dir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc323.0K/mc_data/batch0/cm301726k_si_004_clean
sub dir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc323.0K/mc_data/batch0/ESIWAF_clean
sub dir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc323.0K/mc_data/batch0/PUWCIT_clean
sub dir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc323.0K/mc_data/batch0/XEDLUN01_clean
sub dir: /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc323.0K/mc_data/batch0/TAMZUE_char

In [16]:
for sub_gcmc_workdir in gcmc_workdir.glob("batch*"):
    gcmc_pre_task_submit(sub_gcmc_workdir, n_cpus=32, nodelist=None, job_name=f"gcmc_pre_ddmof_{sub_gcmc_workdir.name}")

Submitted batch job 200744
Submitted batch job 200745
Submitted batch job 200746
Submitted batch job 200747
Submitted batch job 200748
Submitted batch job 200749
Submitted batch job 200750
Submitted batch job 200751
Submitted batch job 200752
Submitted batch job 200753
Submitted batch job 200754
Submitted batch job 200755
Submitted batch job 200756
Submitted batch job 200757
Submitted batch job 200758
Submitted batch job 200759
Submitted batch job 200760
Submitted batch job 200761
Submitted batch job 200762
Submitted batch job 200763
Submitted batch job 200764
Submitted batch job 200765
Submitted batch job 200766
Submitted batch job 200767
Submitted batch job 200768
Submitted batch job 200769
Submitted batch job 200770
Submitted batch job 200771
Submitted batch job 200772
Submitted batch job 200773
Submitted batch job 200774
Submitted batch job 200775
Submitted batch job 200776
Submitted batch job 200777
Submitted batch job 200778
Submitted batch job 200779
Submitted batch job 200780
S

In [8]:
for sub_gcmc_workdir in gcmc_workdir.glob("batch*"):
    unfinished_dirs_charge_eq, unfinished_dirs_void_fraction = run_and_check_pre_tasks(sub_gcmc_workdir, interval=10,
                            num_interval = 1,
                            executor = "bash",
                            check_only = True)

Checking if all charge equilibration simulations are finished...
100/100
charge equilibration simulations all done!
Checking if all helium void fraction simulations are finished...
100/100
helium void fraction simulations all done!
Checking if all charge equilibration simulations are finished...
100/100
charge equilibration simulations all done!
Checking if all helium void fraction simulations are finished...
100/100
helium void fraction simulations all done!
Checking if all charge equilibration simulations are finished...
99/100
charge equilibration simulations not finished after 0.16666666666666666 minites, force stop!
number of unfinished simulations: 1
information of unfinished simulations saved in /home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_4606_mc298.0K/mc_data/batch38/unfinished_dirs_charge_eq_20241120235955.txt
Checking if all helium void fraction simulations are finished...
100/100
helium void fraction simulations all done!
Checking if all charge e

In [9]:
for sub_gcmc_workdir in gcmc_workdir.glob("batch*"):
    gcmc_task_submit(sub_gcmc_workdir, n_cpus=32, nodelist=None, job_name=f"gcmc_ddmof_{sub_gcmc_workdir.name}")

Submitted batch job 200882
Submitted batch job 200883
Submitted batch job 200884
Submitted batch job 200885
Submitted batch job 200886
Submitted batch job 200887
Submitted batch job 200888
Submitted batch job 200889
Submitted batch job 200890
Submitted batch job 200891
Submitted batch job 200892
Submitted batch job 200893
Submitted batch job 200894
Submitted batch job 200895
Submitted batch job 200896
Submitted batch job 200897
Submitted batch job 200898
Submitted batch job 200899
Submitted batch job 200900
Submitted batch job 200901
Submitted batch job 200902
Submitted batch job 200903
Submitted batch job 200904
Submitted batch job 200905
Submitted batch job 200906
Submitted batch job 200907
Submitted batch job 200908
Submitted batch job 200909
Submitted batch job 200910
Submitted batch job 200911
Submitted batch job 200912
Submitted batch job 200913
Submitted batch job 200914
Submitted batch job 200915
Submitted batch job 200916
Submitted batch job 200917
Submitted batch job 200918
S

In [2]:
def density_scatter(x, y, ax=None, is_cbar=False, log_scale=False, **kwargs):
    if ax is None:
        fig, ax = plt.subplots()
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)
    idx = z.argsort()
    x, y, z = x[idx], y[idx], z[idx]
    scatter = ax.scatter(x, y, c=z, cmap='Spectral_r', **kwargs)
    if is_cbar:
        norm = plt.Normalize(vmin=np.min(z), vmax=np.max(z))
        cbar = plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap='Spectral_r'), ax=ax)
        cbar.ax.set_ylabel('Density')
    # Set log scale if needed
    if log_scale:
        ax.set_xscale('log')
        ax.set_yscale('log')
    return ax

def plot_scatter(targets, predictions, title=None, metrics=None, outfile=None, ax=None, log_scale=False):
    targets = np.array(targets)
    predictions = np.array(predictions)
    max_value = max(targets.max(), predictions.max())
    min_value = min(targets.min(), predictions.min())
    offset = (max_value - min_value) * 0.06
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plot density scatter
    density_scatter(targets, predictions, ax=ax, is_cbar=True, log_scale=log_scale)
    
    # Add diagonal line
    ax.plot([min_value, max_value], [min_value, max_value], 'r--')
    
    # Set labels and limits
    ax.set_xlabel("Ground Truth")
    ax.set_ylabel("Predictions")
    ax.set_xlim(min_value - offset, max_value + offset)
    ax.set_ylim(min_value - offset, max_value + offset)
    
    # Set title
    if title:
        ax.set_title(title)
    
    # Display metrics
    if metrics:
        text_content = "\n".join([f"{k}: {v:.4f}" for k, v in metrics.items()])
        ax.text(max_value - offset * 6, min_value + offset, text_content, fontsize=12, color='red')
    
    # Save to file if needed
    if outfile:
        plt.savefig(outfile, dpi=300, bbox_inches='tight', format='png')
    
    return ax

def map_composition(df):
    if df["GasName"] == "CO2":
        return round(df["MoleculeFraction"], 4)
    else:
        return round((1-df["MoleculeFraction"]), 4)

## GCMC results

In [2]:
dataset = "stable_coremof_1225_mc573.0K"
gcmc_workdir = Path(f"CGCNN_MT/inference/CoREMOF2019/{dataset}/mc_data")
result_dir = gcmc_workdir.parent
n_jobs = 32
gases = ["CO2", "N2"]
df_ads = process_isotherm_results(gcmc_workdir, gases, unit="mol/kg", verbose=0, n_jobs=n_jobs)
print("number of isotherm data points: ", len(df_ads))
if len(df_ads) > 0:
    df_ads.to_csv(result_dir/(f'00-isotherm-data-{dataset}.tsv'), index=False, sep='\t', float_format='%.6f')

Found wrong simulation result in CGCNN_MT/inference/CoREMOF2019/stable_coremof_1225_mc573.0K/mc_data/KOGJOG_clean/Adsorption_pure_CO2_0.100bar/Output/System_0/output_KOGJOG_clean_3.3.1_573.000000_10000.data
Found wrong simulation result in CGCNN_MT/inference/CoREMOF2019/stable_coremof_1225_mc573.0K/mc_data/AWASAU_clean/Adsorption_pure_N2_0.100bar/Output/System_0/output_AWASAU_clean_2.4.1_573.000000_10000.data
Found wrong simulation result in CGCNN_MT/inference/CoREMOF2019/stable_coremof_1225_mc573.0K/mc_data/KOGJOG_clean/Adsorption_CO2_N2_0.100_0.900_0.100bar/Output/System_0/output_KOGJOG_clean_3.3.1_573.000000_10000.data
Found wrong simulation result in CGCNN_MT/inference/CoREMOF2019/stable_coremof_1225_mc573.0K/mc_data/KOGJOG_clean/Adsorption_pure_CO2_1.000bar/Output/System_0/output_KOGJOG_clean_3.3.1_573.000000_100000.data
Found wrong simulation result in CGCNN_MT/inference/CoREMOF2019/stable_coremof_1225_mc573.0K/mc_data/AWASAU_clean/Adsorption_pure_N2_1.000bar/Output/System_0/outp

/home/zhangsd/repos/CF-BGAP/GCMC/utils.py:782: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs)


In [3]:
# print(df_ads.MofName.unique())
print(df_ads.MofName.unique().shape)

(1162,)


In [10]:
ads_all = []
valid_cols = ["MofName"]
ads_keys = []
for p in [0.1, 1]:
    for gas in ["CO2", "N2"]:
        sub_df_mix = df_ads[(df_ads["Pressure[bar]"]==p)&(df_ads["MoleculeFraction"].isin([0.1, 0.9]))&(df_ads["GasName"]==gas)]
        sub_df_pure = df_ads[(df_ads["Pressure[bar]"]==p)&(df_ads["MoleculeFraction"].isin([0, 1]))&(df_ads["GasName"]==gas)]
        sub_df_mix = sub_df_mix.rename(columns={"AbsLoading": f"Ads{gas}_{p}bar_mix0.1"})
        sub_df_pure = sub_df_pure.rename(columns={"AbsLoading": f"Ads{gas}_{p}bar_pure"})
        sub_df_merge = pd.merge(sub_df_mix[valid_cols+[f"Ads{gas}_{p}bar_mix0.1"]], 
                                 sub_df_pure[valid_cols+[f"Ads{gas}_{p}bar_pure"]], on=["MofName"], how="outer")
        ads_all.append(sub_df_merge)
        ads_keys.append(f"Ads{gas}_{p}bar_mix0.1")
        ads_keys.append(f"Ads{gas}_{p}bar_pure")
ads_all_df = ads_all[0]
for i in range(1, len(ads_all)):
    ads_all_df = pd.merge(ads_all_df, ads_all[i], on=["MofName"], how="outer")
print(ads_all_df.shape)
ads_all_df

(1162, 9)


,MofName,AdsCO2_0.1bar_mix0.1,AdsCO2_0.1bar_pure,AdsN2_0.1bar_mix0.1,AdsN2_0.1bar_pure,AdsCO2_1bar_mix0.1,AdsCO2_1bar_pure,AdsN2_1bar_mix0.1,AdsN2_1bar_pure
0,ABETIN_clean,0.000722,0.010083,0.004500,0.004861,0.009778,0.100612,0.044501,0.048917
1,ACECIV_ion_b,0.368162,0.400512,0.000232,0.001390,0.402327,0.472386,0.000320,0.013659
2,ADASIJ_clean,0.001006,0.009547,0.004108,0.004615,0.009540,0.097767,0.040767,0.047545
3,ADAVIM_clean,0.000894,0.007708,0.003403,0.003788,0.007996,0.079460,0.034457,0.037552
4,AFIXAO_clean,0.000427,0.004156,0.001394,0.001450,0.003915,0.040033,0.014360,0.015577
...,...,...,...,...,...,...,...,...,...
1157,jacs.6b09113_ja6b09113_si_002_clean,0.001023,0.009683,0.004113,0.004447,0.010094,0.099797,0.039221,0.044056
1158,jacs.6b09155_ja6b09155_si_009_clean,0.000794,0.007845,0.003031,0.003309,0.007134,0.074575,0.028185,0.030927
1159,jp102463p_si_002_clean,0.000795,0.007860,0.002307,0.002910,0.007487,0.073260,0.025314,0.029229
1160,jz4002345_si_002_clean,0.000570,0.006459,0.001745,0.001950,0.006098,0.059934,0.017855,0.019154


In [11]:
col_list = [
    'MofName',
    'AdsCO2_0.1bar_pure',
    'AdsN2_0.1bar_pure',
    'AdsCO2_0.1bar_mix0.1',
    'AdsN2_0.1bar_mix0.1',
    'AdsCO2_1bar_pure',
    'AdsN2_1bar_pure',
    'AdsCO2_1bar_mix0.1',
    'AdsN2_1bar_mix0.1',
    ]
for p in ['0.1bar', '1bar']:
    for m in ['pure','mix0.1']:
        ads_all_df[f"S_{p}_{m}"] = ads_all_df[f'AdsCO2_{p}_{m}']/(ads_all_df[f'AdsN2_{p}_{m}']+1e-10)
        col_list.append(f"S_{p}_{m}")

for gas in ['CO2', 'N2']:
    for m in ['mix0.1']:
        ads_all_df[f"WC_{gas}_{m}"] = ads_all_df[f'Ads{gas}_1bar_{m}']-ads_all_df[f'Ads{gas}_0.1bar_{m}']
        col_list.append(f"WC_{gas}_{m}")

ads_all_df = ads_all_df.reindex(columns=col_list)
col_list

['MofName',
 'AdsCO2_0.1bar_pure',
 'AdsN2_0.1bar_pure',
 'AdsCO2_0.1bar_mix0.1',
 'AdsN2_0.1bar_mix0.1',
 'AdsCO2_1bar_pure',
 'AdsN2_1bar_pure',
 'AdsCO2_1bar_mix0.1',
 'AdsN2_1bar_mix0.1',
 'S_0.1bar_pure',
 'S_0.1bar_mix0.1',
 'S_1bar_pure',
 'S_1bar_mix0.1',
 'WC_CO2_mix0.1',
 'WC_N2_mix0.1']

In [13]:
ads_all_df.sort_values(by=['WC_CO2_mix0.1'], ascending=False)

,MofName,AdsCO2_0.1bar_pure,AdsN2_0.1bar_pure,AdsCO2_0.1bar_mix0.1,AdsN2_0.1bar_mix0.1,AdsCO2_1bar_pure,AdsN2_1bar_pure,AdsCO2_1bar_mix0.1,AdsN2_1bar_mix0.1,S_0.1bar_pure,S_0.1bar_mix0.1,S_1bar_pure,S_1bar_mix0.1,WC_CO2_mix0.1,WC_N2_mix0.1
67,BUWMAJ_clean,7.750256,0.435254,4.333420,0.002091,9.923891,1.955301,7.279241,0.012011,17.806271,2072.304279,5.075379,606.034476,2.945821,0.009920
280,GEBCAT_clean,2.784860,0.002983,1.023635,0.002127,3.523731,0.029592,2.866412,0.007065,933.561100,481.164813,119.076947,405.727326,1.842777,0.004937
831,TUYJED_clean,6.499362,0.024891,4.903026,0.002909,7.753453,0.230967,6.674379,0.018623,261.109780,1685.251703,33.569554,358.394085,1.771353,0.015714
865,VATYAR_clean,9.747610,3.126145,7.906500,0.168785,10.991489,3.159469,9.577707,0.099568,3.118093,46.843726,3.478904,96.192787,1.671207,-0.069217
505,MAKRIC_clean,2.480664,0.005437,1.031128,0.003681,3.723340,0.051394,2.630055,0.017837,456.239344,280.145331,72.447103,147.452998,1.598928,0.014156
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6,AJOXAA_clean,3.798395,3.752363,3.541180,0.220628,4.041632,3.761193,3.492223,0.315789,1.012267,16.050446,1.074561,11.058729,-0.048957,0.095161
902,WAGZEL_clean,2.250445,2.244954,2.058579,0.187081,2.296817,2.245282,1.967646,0.280751,1.002446,11.003650,1.022953,7.008513,-0.090933,0.093669
438,KIHTOL_clean,2.081237,2.081198,1.994482,0.086717,2.081277,2.081198,1.864407,0.216792,1.000019,23.000000,1.000037,8.600000,-0.130075,0.130075
437,KIHTOL01_clean,2.081389,2.081198,1.997040,0.083972,2.081211,2.081198,1.864407,0.216805,1.000092,23.782207,1.000006,8.599484,-0.132633,0.132832


In [17]:
ads_all_df.sort_values(by=['WC_CO2_mix0.1'], ascending=False).to_excel(result_dir/f"{dataset}_ads.xlsx", index=False)